<a href="https://colab.research.google.com/github/mkane968/digital-text-methods/blob/main/Tutorial_7_Simple_Topic_Modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tutorial 7: Simple Topic Modeling

**Topic modeling** is an exploratory method for identifying groups of words that tend to occur together across a collection of documents.

Rather than telling a computer what topics to look for, we ask a model to identify recurring patterns in the corpus.

In this tutorial, we will use **Latent Dirichlet Allocation (LDA)** to build a very simple topic model.

### In this tutorial, we will:
- create a document-term matrix
- choose a number of topics
- fit an LDA model
- inspect the words associated with each topic
- examine which topics appear in each document
- consider the interpretive limits of topic models


## 1. Create a Small Example Corpus

Topic modeling normally works best with substantially more text than we use here. This small dataset is designed only to make the process easy to see.


In [1]:
import pandas as pd

labels = [
    "garden", "garden", "garden",
    "sea", "sea", "sea",
    "city", "city", "city"
]

texts = [
    "flowers garden trees birds spring green leaves",
    "garden flowers trees soil plants birds summer",
    "trees flowers garden leaves plants green field",
    "sea waves ship ocean shore wind water",
    "ocean ship waves sailors water sea storm",
    "shore sea water waves ship wind harbor",
    "city street crowd building traffic shop people",
    "street city buildings people crowd market traffic",
    "people traffic street shops city crowd building"
]

df = pd.DataFrame(
    list(zip(labels, texts)),
    columns=["example_group", "text"]
)

df


,example_group,text
0,garden,flowers garden trees birds spring green leaves
1,garden,garden flowers trees soil plants birds summer
2,garden,trees flowers garden leaves plants green field
3,sea,sea waves ship ocean shore wind water
4,sea,ocean ship waves sailors water sea storm
5,sea,shore sea water waves ship wind harbor
6,city,city street crowd building traffic shop people
7,city,street city buildings people crowd market traffic
8,city,people traffic street shops city crowd building


The `example_group` column is included so that we know what these toy documents are about. **The topic model will not use that column.**

It will receive only the texts.


## 2. Turn Text into Numbers

Machine-learning models cannot work directly with sentences as humans read them.

We first create a **document-term matrix**: a table representing how often words occur in each document.

Scikit-learn's `CountVectorizer` can do this for us.


In [2]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(stop_words="english")

X = vectorizer.fit_transform(df["text"])


We can see the words represented in the matrix:


In [3]:
vectorizer.get_feature_names_out()


array(['birds', 'building', 'buildings', 'city', 'crowd', 'field',
       'flowers', 'garden', 'green', 'harbor', 'leaves', 'market',
       'ocean', 'people', 'plants', 'sailors', 'sea', 'ship', 'shop',
       'shops', 'shore', 'soil', 'spring', 'storm', 'street', 'summer',
       'traffic', 'trees', 'water', 'waves', 'wind'], dtype=object)

And its shape:


In [4]:
X.shape


(9, 31)

The first number is the number of documents. The second is the number of unique words represented as features.


## 3. Build an LDA Topic Model

We will ask the model to identify **3 topics**.

The number of topics is something the researcher chooses; the model does not automatically know the correct number.


In [5]:
from sklearn.decomposition import LatentDirichletAllocation

lda = LatentDirichletAllocation(
    n_components=3,
    random_state=42
)

lda.fit(X)


LatentDirichletAllocation(n_components=3, random_state=42)

`random_state=42` makes the example reproducible so that we receive the same results when we rerun the notebook.


## 4. Examine the Topics

A topic is represented by a group of words with different weights.

Let's display the five highest-weighted words for each topic:


In [6]:
feature_names = vectorizer.get_feature_names_out()

for topic_number, topic in enumerate(lda.components_):
    top_words = topic.argsort()[-5:][::-1]
    words = [feature_names[i] for i in top_words]
    print("Topic", topic_number + 1, ":", words)


Topic 1 : ['waves', 'water', 'ship', 'sea', 'shore']
Topic 2 : ['traffic', 'street', 'people', 'crowd', 'city']
Topic 3 : ['trees', 'garden', 'flowers', 'leaves', 'green']


The model gives us **words**, not meaningful topic names.

Humans interpret the word groups and decide whether labels such as `"garden"`, `"sea"`, or `"city"` are useful descriptions.

Topic labels are therefore **interpretations made by researchers**, not facts produced by the model.


## 5. Topic Mixtures

LDA assumes that a document can contain a mixture of topics.

Let's calculate the topic distribution for each document:


In [7]:
topic_distributions = lda.transform(X)

topic_distributions


array([[0.04180121, 0.04180121, 0.91639758],
       [0.04182278, 0.04182278, 0.91635445],
       [0.04180121, 0.04180121, 0.91639758],
       [0.91645388, 0.04177306, 0.04177306],
       [0.91636791, 0.04181604, 0.04181605],
       [0.91641098, 0.04179451, 0.04179451],
       [0.04178781, 0.91642438, 0.04178781],
       [0.04180931, 0.91638137, 0.04180932],
       [0.04178781, 0.91642438, 0.04178781]])

The numbers represent the model's estimated distribution of topics within each document.

We can add the strongest topic to our DataFrame:


In [8]:
df["dominant_topic"] = topic_distributions.argmax(axis=1) + 1

df


,example_group,text,dominant_topic
0,garden,flowers garden trees birds spring green leaves,3
1,garden,garden flowers trees soil plants birds summer,3
2,garden,trees flowers garden leaves plants green field,3
3,sea,sea waves ship ocean shore wind water,1
4,sea,ocean ship waves sailors water sea storm,1
5,sea,shore sea water waves ship wind harbor,1
6,city,city street crowd building traffic shop people,2
7,city,street city buildings people crowd market traffic,2
8,city,people traffic street shops city crowd building,2


Compare the model's dominant topics with the `example_group` column.

Because this example was intentionally constructed around three very distinct vocabularies, the patterns may appear unusually clear. Real literary corpora are much messier.


## 6. A More Useful Topic Table

Let's place each document's topic weights into the DataFrame.


In [9]:
for i in range(3):
    df[f"topic_{i+1}"] = topic_distributions[:, i]

df


,example_group,text,dominant_topic,topic_1,topic_2,topic_3
0,garden,flowers garden trees birds spring green leaves,3,0.041801,0.041801,0.916398
1,garden,garden flowers trees soil plants birds summer,3,0.041823,0.041823,0.916354
2,garden,trees flowers garden leaves plants green field,3,0.041801,0.041801,0.916398
3,sea,sea waves ship ocean shore wind water,1,0.916454,0.041773,0.041773
4,sea,ocean ship waves sailors water sea storm,1,0.916368,0.041816,0.041816
5,sea,shore sea water waves ship wind harbor,1,0.916411,0.041795,0.041795
6,city,city street crowd building traffic shop people,2,0.041788,0.916424,0.041788
7,city,street city buildings people crowd market traffic,2,0.041809,0.916381,0.041809
8,city,people traffic street shops city crowd building,2,0.041788,0.916424,0.041788


This lets us examine topic patterns alongside metadata and eventually compare topic prevalence across authors, periods, genres, or other categories.


Let's visualize the topics.

In [10]:
!pip install pyLDAvis -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 63.3 MB/s eta 0:00:00


In [11]:
import pyLDAvis
import pyLDAvis.lda_model

visualization = pyLDAvis.lda_model.prepare(
    lda,
    X,
    vectorizer
)

pyLDAvis.display(visualization)

/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.13/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


**Reading the Topic Visualization**

Each circle represents one topic. Larger circles represent topics that account for more of the corpus. Topics that appear closer together use more similar vocabularies. Topics farther apart are more distinct. Clicking a topic changes the word list on the right to show the terms most strongly associated with that topic.

In this example, the three topics are relatively far apart because the sample corpus was deliberately created around three distinct sets of vocabulary: urban life, the sea, and gardens/nature. Real literary corpora will usually produce messier and more overlapping topics.

## 7. What Does a Topic Model Actually Tell Us?

A topic model does **not**:

- read a text the way a human reader does
- discover the one correct set of themes
- explain what a text means
- automatically know how many topics exist
- provide objective topic labels

Instead, it identifies **statistical patterns of word co-occurrence across documents**.

Researchers still need to interpret those patterns and return to the texts.


## Try It Yourself

1. Change `n_components=3` to `n_components=2`.
2. Rerun the model and examine the topics.
3. Then try `n_components=4`.
4. Compare the results.

In a text cell, respond:

**How did changing the number of topics change the model's representation of the corpus? Which version seems more interpretable, and what evidence from the words/documents supports your interpretation?**


## From Topic Models to Literary Questions

Topic modeling can be useful for **exploration** when a corpus is too large to read document by document at the beginning of a project.

A common workflow is:

**Documents → Document-Term Matrix → Topic Model → Word Groups → Document Patterns → Return to Texts → Interpret**

Possible questions include:

- Do different periods emphasize different clusters of vocabulary?
- Do genres differ in their topic distributions?
- Which documents appear unusual compared with the rest of the corpus?
- Do patterns identified computationally suggest texts or passages for closer reading?

Topic modeling is most useful when it creates **new questions to investigate**, rather than when it is treated as an automatic answer.


## References and Additional Resources

- **scikit-learn: Latent Dirichlet Allocation**  
  https://scikit-learn.org/stable/modules/decomposition.html#latentdirichletallocation

- **scikit-learn: CountVectorizer**  
  https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.CountVectorizer.html

- **Introduction to Cultural Analytics & Python — Melanie Walsh**  
  https://melaniewalsh.github.io/Intro-Cultural-Analytics/

- **Programming Historian**  
  https://programminghistorian.org/


**AI Use Disclosure:** This tutorial was developed by the instructor with assistance from AI tools for drafting and refining explanations, examples, and Python code. All materials were reviewed, edited, and adapted by the instructor for this course.